# Публичная копия исследовательского ноутбука

Исторические агрегаты относятся к исследованию 6 октября, не к запуску GitHub. Выводы, вложения и таблицы с точками удалены. Для чтения разрешённого исходного 89-X набора задайте TRAFFIC_RESEARCH_DATA. Полный 155/180-колоночный срез описан отдельно; это разные наборы. Естественность не подтверждена.


# Дополнение: обычные HTTPS-клиенты

Добавлены **12 независимых захватов**. Chrome и curl прочитали example.org, docs.python.org и Wikipedia через существующий HTTP-прокси без авторизации. Проверены успешные ответы и содержимое страниц; uploads и искусственный polling отсутствуют. Это настоящие клиентские стеки; их поведение не объявляется реальным офисным использованием. Scope: `real Chrome/curl reads of three public HTTPS pages through existing unauthenticated HTTP proxy; logical endpoint mapping; not direct Internet or observed office usage`. Сырые захваты сохранены. Для публичного импорта применяется отдельная проекция порядка: только регрессии ≤10 мкс между направлениями/потоками; байты, временные метки и порядок каждого TCP-направления проверяются точно. Файлов с перестановками: 1 из 12; переставлено записей: 5; максимальная регрессия: 5.007 мкс. Подробные исходные и производные хеши сохранены в отчёте.

Ранее добавленные обычные сегменты: **194**, они сохранены точно. Все эти строки имеют `label_binary=0`, `label_state=hard_negative`.

Новых строк после размещения в двух днях: **214**. По клиентам: `{'BENIGN_PUBLIC_CHROME': 202, 'BENIGN_PUBLIC_CURL': 12}`. Split: `{'validation': 78, 'train': 12, 'test': 124}`. Исходные размеченные признаки, метки и группы train/test сохранены точно; офисный фон в итоговой таблице появляется один раз. Новые контроли прошли общий адаптер `activity.json`, а не специальный импорт Cover Channels.

Отдельная проверка новых обычных клиентов против исходящего офисного TCP/TLS: **AUC 0.4997**, строки `{'train_control': 6, 'test_control': 42, 'train_office': 5000, 'test_office': 5000}`. Это небольшая оценка с удержанными профилями и повторным размещением тех же захватов; её нельзя считать доказательством production-переноса. `None` означает отсутствие достаточных групп для оценки. В этой проверке всего 6 generated-строк на train против 5000 офисных. AUC около 0,5 здесь может отражать недостаточную мощность классификатора и дисбаланс; это не подтверждение сходства. В частности, у диагностического HistGradientBoosting min_samples_leaf=20.

Модель наблюдения сенсора для этого дополнения: `sensor_capacity_complete=True`; пакетов сверх эмпирической цели 8: **0**. Версии наблюдения: `['office-stamps-v4']`. Дополнительная микросекундная метка — ограниченная аппроксимация, механизм сенсора не воспроизведён. При false дополнение остаётся диагностическим и не считается прошедшим проверку естественности.

Прежняя проверка matched-control против офиса ниже сохраняет прежнее определение популяции. Добавление новых негативов не делает старые контроли естественными и не позволяет скрыть их AUC усреднением.

Метрики обнаружения до дополнения: `{'roc_auc': 0.6872480756801223, 'average_precision': 0.6838779463421282, 'balanced_accuracy_at_050': 0.627651448381221, 'rows': 1168}`; после: `{'roc_auc': 0.7225811334759212, 'average_precision': 0.6728647172884283, 'balanced_accuracy_at_050': 0.6579038606154717, 'rows': 1292}`. Состав отрицательного теста изменился, поэтому это не парная оценка причинного улучшения. Хеш подтверждений дополнения: `6e6d4529b3386676b7ff968c22ec3854df73f9c3f94d26c2a478db3464aa62b6`. Первичные seals Cover Channels и отдельные подтверждения обычного дополнения сохраняются.


# Проверка мощности диагностики происхождения

Низкий AUC не доказывает естественность, если классификатор почти не видел примеров одного источника. Ниже сравниваются обычный HGB и три модели с балансировкой классов. Все 89 признаков остаются; HGB сохраняет наблюдаемые пропуски, остальные модели заполняют их медианой только train. Пропуски сами могут быть признаком происхождения.

Офисные части разделены по идентификаторам сессий: одна сессия не попадает в разные части. Generated-части сохраняют исходное разбиение по профилям. Число сегментов не заменяет число независимых групп. Проверка не утверждает, что офис целиком доброкачественный.

| Популяция | Generated train / test | Группы train / test | HGB обычный | HGB balanced | ExtraTrees balanced | Logistic balanced | Поддержка |
|---|---:|---:|---:|---:|---:|---:|---|
| original_matched_controls | 607 / 183 | 119 / 37 | 0.9999912568306011 | 0.9999836065573771 | 0.9998 | 0.9995060109289617 | sufficient_segment_count |
| ordinary_fixture | 56 / 17 | 5 / 2 | 0.9999882352941176 | 1.0 | 0.9999764705882352 | 0.9971529411764706 | insufficient_segments |
| public_first | 6 / 42 | 3 / 2 | 0.9521095238095238 | 0.9998238095238094 | 1.0 | 0.9999571428571428 | insufficient_generated_train |

Контроль мощности: случайное разделение офисных сессий на два псевдоисточника — `{'feature_order': ['pkt_count', 'up_pkt_count', 'down_pkt_count', 'total_bytes', 'up_bytes', 'down_bytes', 'flow_duration', 'up_down_pkt_ratio', 'up_down_bytes_ratio', 'pkt_rate', 'byte_rate', 'pkt_len_mean', 'pkt_len_std', 'pkt_len_min', 'pkt_len_max', 'pkt_len_median', 'pkt_len_p10', 'pkt_len_p90', 'pkt_len_p95', 'pkt_len_entropy', 'iat_mean', 'iat_std', 'iat_min', 'iat_max', 'iat_p50', 'iat_p90', 'iat_p99', 'iat_entropy', 'iat_cv', 'iat_regularity', 'burst_count', 'idle_ratio', 'direction_changes', 'syn_count', 'fin_count', 'rst_count', 'psh_count', 'ack_count', 'len_entropy_up', 'len_entropy_down', 'len_unique_up', 'len_unique_down', 'small_pkt_share', 'low_rate_long', 'dir_entropy', 'idle_gt60_count', 'up_bytes_per_pkt', 'down_bytes_per_pkt', 'const_len_share_up', 'burst_len_mean', 'burst_len_max', 'dir_burst_count', 'dir_burst_len_mean', 'dir_burst_len_max', 'ra_small_up_count', 'ra_small_up_share', 'ra_keystroke_iat_p50', 'ra_echo_ratio', 'ra_echo_latency_p50', 'ra_interactive_score', 'ra_burst_after_idle', 'tcp_handshake_rtt_ms', 'data_pkt_up', 'data_pkt_down', 'small_data_up_bytes', 'pay_entropy_up', 'pay_entropy_down', 'pay_printable_up', 'pay_printable_down', 'pay_null_share_up', 'pay_b64_share_up', 'pay_bytes_sampled_up', 'pay_bytes_sampled_down', 'tcp_retx_pkts_up', 'tcp_retx_pkts_down', 'tcp_retx_bytes_up', 'tcp_retx_bytes_down', 'tls_sni_len', 'tls_version', 'tls_cipher_count', 'tls_ext_count', 'tls_group_count', 'tls_sigalg_count', 'tls_alpn_h2', 'tls_alpn_h3', 'tls_alpn_http11', 'tls_alpn_other', 'tls_resumed', 'tls_early_data'], 'rows': {'train_generated': 2548, 'train_office': 2452, 'test_generated': 2509, 'test_office': 2491}, 'support_status': 'sufficient_segment_count', 'missingness_policy': 'HGB uses observed NaN; train-empty columns fixed to zero; extra trees/logistic use train-only median imputation; every column retained', 'legacy_unweighted_hgb': {'auc': 0.4775901895688568, 'symmetric_auc': 0.5224098104311432, 'score_std': 0.09562797752955647, 'unique_scores': 4978}, 'balanced_hgb': {'auc': 0.4768148995210978, 'symmetric_auc': 0.5231851004789022, 'score_std': 0.10358617041875233, 'unique_scores': 4974}, 'balanced_extra_trees': {'auc': 0.49127716695208373, 'symmetric_auc': 0.5087228330479163, 'score_std': 0.1045139083463989, 'unique_scores': 4997}, 'balanced_logistic': {'auc': 0.4957701051805631, 'symmetric_auc': 0.5042298948194369, 'score_std': 0.06708670201454893, 'unique_scores': 5000}}`.

Эта проверка использует TCP с SYN, данными в обе стороны и SNI. В компактном офисном экспорте нет private/public-контекста, поэтому её популяция отличается от прежней «более сопоставимой» проверки ниже. Изменение AUC нельзя приписывать только балансировке. Фон уже изучался; это диагностический аудит, а не новый независимый production-тест.

`naturalness_established=False`, `production_ready=False`. Малую поддержку и различие приложений нельзя устранить усреднением источников или удачным видом трёхмерного графика.


# Эксперимент с запретом признаков

Все 89 колонок в Parquet сохранены. Проверено **24 масок**: группы признаков и их исключения, несколько пар исключений, train-only отбор по маргинальной отличимости источника. Это ограниченный поиск, а не доказательство отсутствия любой полезной комбинации.

Критерий исследовательского выбора заранее записан в отчёте: symmetric origin AUC ≤0.60 и technique AUC ≥0.70 на validation одновременно. Выбрана маска: **ни одна**. Маска с наименьшей отличимостью источника: `only_retransmission` — это диагностический вариант, автоматически пригодным он не становится.

Test используется только для замороженного baseline, лучшего по происхождению диагностического варианта и выбранного варианта. Эти test-дни уже анализировались исторически, поэтому эксперимент не заменяет нового независимого теста. Офисная validation разделена по строкам сегментов; независимость сессий не установлена, потому что в метаданных офисной выборки нет идентификатора сессии. Origin сравнивает прежние matched-control с офисом; новые обычные негативы не разбавляют эту оценку.

| Маска | Признаков | Technique AUC validation | Origin AUC validation |
|---|---:|---:|---:|
| all_89 | 89 | 0.6600190912019405 | 1.0 |
| only_direction | 11 | 0.5269171735546586 | 0.9940482073244385 |
| without_direction | 78 | 0.6791706933469767 | 1.0 |
| only_size | 22 | 0.6473338639076801 | 1.0 |
| without_size | 67 | 0.6229890284804729 | 0.9998352824905288 |
| only_timing | 26 | 0.553623538551911 | 0.998045352220941 |
| without_timing | 63 | 0.6762051617304303 | 1.0 |
| only_flags | 5 | 0.561920868626002 | 0.993576017130621 |
| without_flags | 84 | 0.6549134974323991 | 1.0 |
| only_handshake_rtt | 1 | 0.5591010080912587 | 0.8202547630813155 |
| without_handshake_rtt | 88 | 0.6792772819434181 | 1.0 |
| only_payload | 8 | 0.6347753586114111 | 0.989908307253061 |
| without_payload | 81 | 0.6434137721941144 | 1.0 |
| only_retransmission | 4 | 0.5009415326018987 | 0.5 |
| without_retransmission | 85 | 0.6600190912019405 | 1.0 |
| only_tls | 12 | 0.5640929073579292 | 0.9475539449843519 |
| without_tls | 77 | 0.664523051560457 | 1.0 |
| without_size_timing | 41 | 0.6453477630606561 | 0.9997803766540384 |
| without_size_tls | 55 | 0.6551562825687378 | 0.9997254708175479 |
| without_timing_tls | 51 | 0.6699993367820666 | 1.0 |
| without_payload_tls | 69 | 0.6517608436132112 | 1.0 |
| marginal_source_max_0.55 | 34 | 0.5716310897618101 | 0.998292428485148 |
| marginal_source_max_0.6 | 46 | 0.6307072271805658 | 1.0 |
| marginal_source_max_0.65 | 50 | 0.6357701855115305 | 0.9999999999999999 |

Удержанные результаты: `{'all_89': {'test_technique_auc': 0.6687090426103347, 'test_origin_auc': 0.9988758255656003}, 'only_retransmission': {'test_technique_auc': 0.5051248620511168, 'test_origin_auc': 0.5}}`. Снижение отличимости источника ценой потери сигнала техники не считается успехом. Production-ready остаётся false.

# Cover Channels: офис, все механики и настоящий Adaptix

**Результат: естественность по диагностике происхождения не достигнута.** Все 89 признаков сохранены. Можно воспроизвести исследовательское обучение; переносимость на живой офис пока не проверена.

Этот ноутбук описывает новую, запечатанную версию набора, а не старые результаты 4 октября. Начните с разделов 1–5; ниже приведены все механики, словарь признаков и код чтения Parquet.

**Одна строка — сегмент сессии, а не пакет.** Размеченных строк: 6,024; офисная выборка для графика: 10,000; сохранённый фон: 401,835 сегментов. Один запуск может дать несколько строк. Повторное размещение того же захвата в двух днях не создаёт независимую активность.


## 1. Как получились данные

1. Взяли сохранённый офисный фон двух дней. Офисные строки не объявляются benign: их истинные метки неизвестны.
2. Для каждой механики есть сценарий и обычный контроль. Обе роли получают одинаковый сетевой путь. Фиксируются исходный захват, проверка выполнения и хеши.
3. Сценарий и контроль подмешиваются в отдельные альтернативы одного фона. В общей таблице чистый офис учитывается один раз, добавляются только размеченные строки альтернатив.
4. Перед извлечением признаков к новым пакетным строкам применяется модель наблюдения: время сдвигается вперёд к подходящей офисной пачке не более чем на 500 мкс. Размеры, данные, порядок пакетов и исходные PCAP сохраняются. Отдельный интервал может измениться до 500 мкс; это не утверждение о точном сохранении миллисекундных интервалов.
5. Один и тот же код извлечения считает все 89 признаков. Разметка, адреса, порты, происхождение, split и cluster хранятся отдельно от X.

Это исследовательская композиция сохранённых пакетных строк. Она не равна физическому запуску агента через офисный сенсор.


## 2. Что изменили и что проверили

Путь `office_path_v1`: MTU клиента 1290 (MSS TCP 1250), распределение RTT и доля TCP timestamps из измеренного офиса. Такой путь относится к исходящему интернет-трафику; весь офис содержит и другие пути. Для сценария и контроля среда одинакова.

Для четырёх захватов `M-L34-STORAGE`, самостоятельно собирающих SYN, исключение SYN-проверки записано явно; проверка размеров кадров остаётся. Два raw/ICMP захвата остаются research-only и не дают обычных транспортных строк для обучения.

Пачечность подтверждена чтением сырых PCAP: 76% соседних пакетов имеют одинаковое время; группы до 8 пакетов часто содержат несколько потоков. Это наблюдаемый эффект захвата, а точная причина в драйвере/кольце/опросе не доказана. Модель приближает наблюдение и не меняет схему признаков.

Проверяются побитовая неизменность всех полей кроме времени, порядок, предел сдвига, ёмкость пачек, сохранность офисных строк, соответствие GT извлечённым строкам, исходные PCAP-хеши и запечатанные манифесты. Нулевой сдвиг не освобождает занятое место в пачке. Время payload берётся из фактического отображения, а не выделяется повторно.


| Ветка | Пакеты на офисных метках | Переполнение | Проверка ёмкости завершена |
|---|---:|---:|---|
| composition_day22_v2 / scenario | 27290 | 0 | True |
| composition_day22_v2 / control | 27317 | 0 | True |
| composition_day28_v2 / scenario | 27287 | 0 | True |
| composition_day28_v2 / control | 27349 | 0 | True |

## 3. Настоящий Adaptix: что выполнено

12 захватов: TCP и mTLS, по три профиля, сценарий и обычная Go-телеметрия. В сценариях проверены 36 фиксированных задач чтения подготовленных файлов. Реально исполнены регистрация, постоянное соединение, AES-GCM framing, task/result exchanges; mTLS добавляет взаимную проверку сертификатов.

Upstream-код, задания, содержимое файлов и расписание совпадают с прежним запуском. Старые PCAP проверены по хешам. Новый сетевой путь меняет сегментацию/сетевые интервалы; новые ключи и сессии также означают новые байты нового исполнения. Модель наблюдения не переписывает исходные PCAP.

Mythic не запускался. Другие возможности Adaptix — pivot, persistence и произвольные задания — не считаются выполненными. Раздел показывает проверенное покрытие, а не весь функционал framework.


| Транспорт / профиль | Роль | Пакеты | Проверено задач | Потери захвата |
|---|---|---:|---:|---:|
| tcp / 0 | scenario | 28 | 6 | 0 |
| tcp / 0 | control | 25 | 6 | 0 |
| tcp / 1 | scenario | 39 | 6 | 0 |
| tcp / 1 | control | 37 | 6 | 0 |
| tcp / 2 | scenario | 88 | 6 | 0 |
| tcp / 2 | control | 59 | 6 | 0 |
| mtls / 0 | scenario | 39 | 6 | 0 |
| mtls / 0 | control | 40 | 6 | 0 |
| mtls / 1 | scenario | 45 | 6 | 0 |
| mtls / 1 | control | 52 | 6 | 0 |
| mtls / 2 | scenario | 70 | 6 | 0 |
| mtls / 2 | control | 76 | 6 | 0 |

## 4. Как проверяется естественность

Классификатор происхождения отличает обычные сгенерированные **контроли** от офиса, используя все 89 признаков. Если он уверенно различает источники на других профилях и другом дне, набор содержит различия, которые детектор может выучить вместо целевой активности. AUC около 0,5 означает слабую различимость в данном тесте; AUC около 1 — сильную. Это не FPR детектора.

Train-контроли размещены в train-дне, test-контроли — в test-дне; целые профили остаются в одном split. Страты выделяют сопоставимые сессии по наличию данных в обе стороны и SNI. Диагностика семейств помогает локализовать различие; никакие признаки из итогового X не удалены. Небольшое пересечение по форме активности ограничивает выводы.

Путь и модель наблюдения калиброваны по агрегатам этих офисных дней, включая день оценки. Результат условен на этом фоне и не является независимой проверкой нового офиса. Даже низкий AUC не доказывает, что модель будет работать на production.


### Какие признаки могут объяснять различие

| Группа | Что наблюдаем | Почему этого недостаточно для вывода о канале |
|---|---|---|
| Размеры / направления / объём | Форма запросов и ответов, сегментация, переданный объём | Обычные приложения и MTU тоже меняют эти значения |
| IAT / bursts / idle | Ритм активности и доставки пакетов | Путь и способ timestamping меняют короткие интервалы |
| TLS | Наблюдаемые поля рукопожатия | Часто описывают клиентский стек; не раскрывают команду внутри TLS |
| Payload shape | Статистика доступных байтов | Высокая энтропия встречается в обычном шифрованном трафике |
| RTT / retransmission / flags | Установка соединения, повторы, закрытие | Сильно зависят от сети и границ захвата |
| Interactive / request-answer | Формы коротких обменов и ответов | Администрирование и легитимные приложения тоже дают такие формы |

Все группы остаются в X. Их «решающая» роль требует независимой проверки: важность в одной модели или отличие среднего кластера не устанавливает причинность. Некоторые content-механики после шифрования не обязаны иметь отличимый сигнал в этих 89 сетевых признаках.


| Страта | Train: контроль / офис | Test: контроль / офис | AUC происхождения, все 89 |
|---|---:|---:|---:|
| all | 797 / 7581 | 274 / 2419 | 0.9999864213661313 |
| bidirectional | 791 / 3102 | 274 / 1149 | 0.9999745891381271 |
| bidirectional_sni | 607 / 1555 | 185 / 577 | 0.9988664574453137 |
| bidirectional_sni_volume_matched | 579 / 887 | 176 / 291 | 0.998320837238363 |

### Более сопоставимая популяция

Основная дополнительная проверка: private client → public server, TCP, наблюдавшийся SYN, данные в обе стороны, SNI. Train/test-контексты совпадают по дням, X содержит все 89 признаков. AUC происхождения: **0.9999945355191256**; строки: `{'train_control': 607, 'test_control': 183, 'train_office': 5000, 'test_office': 5000}`.

Ещё более узкая проверка оставляет только общие tuples `(cipher_count, ext_count, group_count, sigalg_count)`, достаточно представленные в обоих **train** источниках. Список tuples заморожен до test; test не выбирает поддержку. Это совпадение четырёх наблюдаемых чисел, а не доказательство идентичности TLS-стека.

Общих tuples: 0; строки: `{'train_control': 0, 'train_office': 0, 'test_control': 0, 'test_office': 0}`; AUC на всех 89: **None**. `None` означает, что оценка недоступна, а не что источники одинаковы. Ограниченная поддержка говорит о недостатке сопоставимых активностей; даже низкий условный AUC не заменяет общую проверку или офисные позитивы.


## 5. Трёхмерные скопления и смысл кластеров

Каждая точка — реальная строка этого экспорта. Показаны все 16,024 строк, дополнительного прореживания нет. График вращается мышью; кнопки меняют цвет: **кластер**, **роль**, **источник** (Adaptix красный), полный диапазон или основные скопления.

У generated-строк split задаётся профилем, у офисной выборки — днём. Train-профиль может присутствовать в обоих офисных контекстах; это условный исследовательский fit, а не независимое испытание нового офиса.

Частоты кластеров относятся к этому смешанному экспорту с офисной выборкой, а не к полной популяции офиса.

Кластеризация: заполнение пропусков → стандартизация → PCA до 10 измерений → KMeans с 8 центрами. Преобразования и центры обучены только на train; test затем назначается ближайшему центру. Число 8 задано заранее и не означает восемь реальных классов активности. Роли и источники используются для объяснения после fit.

График показывает только первые три PCA-компоненты, 34.3% дисперсии. Разные кластеры могут перекрываться в 3D. Их номера произвольны: кластер не означает вредоносность. Все 89 входных признаков сохранены; постоянные/полностью пустые на train колонки не дают геометрической информации.

[Открыть автономный интерактивный график](../reports/cover_channels_natural_clusters_3d_audited_20261006.html).


In [ ]:
# Вывод графика удалён: он содержит точки исходного закрытого набора.


Статический график исключён из публичной копии; постройте его на разрешённом локальном наборе.


### Почему образовались эти группы

Ниже — признаки с самым большим отличием среднего внутри кластера от общего train-среднего, в стандартных отклонениях. Это описание статистической формы, не причинное объяснение. Сравнивайте знак, медиану и состав ролей; редкие выбросы и пропуски могут влиять на среднее.


**C0**: 2904 строк; train 2448, test 401. Роли train: `{'matched_control': 17, 'observed_positive': 22, 'hard_negative': 37, 'unlabelled_office': 2372}`.

| Признак | Значение внутри кластера: медиана | Медиана общего train | Отличие среднего, σ |
|---|---:|---:|---:|
| tls_alpn_other — другой ALPN | 0 | 0 | +4.03 |
| pkt_len_p90 — длина пакета: перцентиль 90 (ближайший элемент ряда, без интерполяции) | 1304 | 340 | +1.50 |
| tls_alpn_h3 — ALPN = HTTP/3 | 0 | 0 | +1.49 |
| pkt_len_p95 — длина пакета: перцентиль 95 (ближайший элемент ряда, без интерполяции) | 1304 | 576 | +1.40 |
| pkt_len_std — длина пакета: разброс | 504.1 | 229.7 | +1.35 |

**C1**: 1767 строк; train 1695, test 54. Роли train: `{'matched_control': 49, 'observed_positive': 54, 'unlabelled_office': 1592}`.

| Признак | Значение внутри кластера: медиана | Медиана общего train | Отличие среднего, σ |
|---|---:|---:|---:|
| pay_null_share_up — доля нулевых байт от клиента | 0.2446 | 0.06771 | +1.13 |
| ra_echo_ratio — доля «нажатий», на которые сервер ответил данными в пределах 1 с (эхо набранного символа); пусто, если ответы сервера зеркалу не видны | 1 | 0.5 | +1.06 |
| iat_entropy — интервал между пакетами: энтропия | 0 | 1.579 | -1.05 |
| ra_small_up_share — доля «нажатий» среди всех пакетов клиента с данными (1 — клиент шлёт только короткие команды) | 1 | 0.2 | +1.05 |
| pkt_rate — пакетов в секунду | 700.5 | 118.6 | +1.01 |

**C2**: 5849 строк; train 3467, test 1361. Роли train: `{'matched_control': 1349, 'hard_negative': 191, 'observed_positive': 1344, 'unlabelled_office': 583}`.

| Признак | Значение внутри кластера: медиана | Медиана общего train | Отличие среднего, σ |
|---|---:|---:|---:|
| len_entropy_down — энтропия длин от сервера | 2.725 | 1.5 | +1.01 |
| pkt_len_entropy — длина пакета: энтропия | 2.725 | 1.973 | +0.86 |
| pay_entropy_down — энтропия первых байт данных от сервера | 7.765 | 5.137 | +0.86 |
| fin_count — число пакетов с флагом FIN | 2 | 1 | +0.80 |
| len_entropy_up — энтропия длин от клиента | 1.947 | 1.379 | +0.74 |

**C3**: 2717 строк; train 2698, test 4. Роли train: `{'matched_control': 6, 'observed_positive': 6, 'unlabelled_office': 2686}`.

| Признак | Значение внутри кластера: медиана | Медиана общего train | Отличие среднего, σ |
|---|---:|---:|---:|
| pay_entropy_up — энтропия первых байт данных от клиента | 0 | 6.18 | -1.45 |
| small_pkt_share — доля кадров короче 100 байт (включая пустые подтверждения TCP) | 1 | 0.6 | +1.22 |
| pay_printable_up — доля печатаемых символов от клиента | 0 | 0.3191 | -1.21 |
| pay_entropy_down — энтропия первых байт данных от сервера | 0 | 5.137 | -1.13 |
| pkt_len_max — длина пакета: максимум | 66 | 1242 | -1.13 |

**C4**: 2 строк; train 1, test 1. Роли train: `{'unlabelled_office': 1}`.

| Признак | Значение внутри кластера: медиана | Медиана общего train | Отличие среднего, σ |
|---|---:|---:|---:|
| burst_len_mean — средняя длина пачки по времени, пакетов (0 — пачек не было) | 2.621e+05 | 5 | +103.09 |
| up_down_bytes_ratio — байты клиент/сервер | 3.542e+08 | 0.9402 | +94.96 |
| up_down_pkt_ratio — пакеты клиент/сервер | 2.621e+05 | 1.125 | +88.91 |
| burst_len_max — самая длинная пачка по времени, пакетов | 2.621e+05 | 7 | +88.44 |
| up_bytes — байт от клиента | 3.542e+08 | 1205 | +86.96 |

**C5**: 2381 строк; train 351, test 1837. Роли train: `{'matched_control': 173, 'observed_positive': 178}`.

| Признак | Значение внутри кластера: медиана | Медиана общего train | Отличие среднего, σ |
|---|---:|---:|---:|
| pay_b64_share_up — доля символов base64 от клиента | 0.8047 | 0.2285 | +2.39 |
| pay_printable_up — доля печатаемых символов от клиента | 0.9289 | 0.3191 | +2.22 |
| pay_printable_down — доля печатаемых символов от сервера | 0.9111 | 0.3477 | +2.08 |
| tls_version — версия TLS | 0 | 5 | -1.97 |
| tls_ext_count — расширений TLS | 0 | 12 | -1.88 |

**C6**: 9 строк; train 9, test 0. Роли train: `{'unlabelled_office': 9}`.

| Признак | Значение внутри кластера: медиана | Медиана общего train | Отличие среднего, σ |
|---|---:|---:|---:|
| dir_burst_count — пачек по направлению: сколько раз подряд шли 2 и больше пакетов в одну сторону (время не учитывается) | 1.292e+04 | 3 | +28.30 |
| direction_changes — смены направления | 2.053e+04 | 6 | +27.31 |
| ack_count — число пакетов с флагом ACK | 1.254e+05 | 11 | +26.80 |
| down_pkt_count — пакетов от сервера | 6.473e+04 | 5 | +25.84 |
| pkt_count — пакетов в сессии | 1.254e+05 | 13 | +25.22 |

**C7**: 395 строк; train 338, test 53. Роли train: `{'unlabelled_office': 338}`.

| Признак | Значение внутри кластера: медиана | Медиана общего train | Отличие среднего, σ |
|---|---:|---:|---:|
| iat_p99 — интервал между пакетами: перцентиль 99 (ближайший элемент ряда, без интерполяции) | 45.14 | 0.04606 | +4.06 |
| iat_max — интервал между пакетами: максимум | 45.77 | 0.04652 | +3.94 |
| iat_std — интервал между пакетами: разброс | 19.69 | 0.0109 | +3.88 |
| flow_duration — длительность сессии, с | 246.9 | 0.118 | +3.71 |
| iat_mean — интервал между пакетами: среднее | 11.55 | 0.009062 | +3.68 |

## 6. Как читать каталог механик

- URI / headers / body / syntax / response: данные или команды располагаются в элементах HTTP. Их наличие в каталоге не означает, что содержимое доступно сенсору внутри TLS.
- Timing: информация связана с интервалами; мелкие интервалы особенно чувствительны к модели наблюдения.
- WebSocket / SSE / longpoll / gRPC / HTTP2 / HTTP3: разные способы длительного или повторяющегося обмена.
- TLS / browser: формы рукопожатия и браузерного обмена; `semantic_fixture` обозначает ограниченный пример семантики, а не полную реализацию браузера.
- Stage M (`M-*`): семейства форм активности с несколькими реализациями клиента. Профили и transport/carrier ниже показывают фактически наблюдавшиеся реализации.
- Framework: настоящий Adaptix task/result; контроль — обычная телеметрия. Hard negative — обычная активность, которая может походить на канал.

Каталог покрывает существующий генератор. Для новой активности нужен отдельный сценарий/контроль, исходный PCAP и подтверждение выполнения; название новой механики само по себе не является подтверждением.


## 6.1. Все проверенные механики

Каждая строка ниже взята из запечатанного реестра. `labelled_rows=0` не скрывается: захват может быть research-only и не давать транспортных признаков. Разные профили одной механики не превращают её в новую механику.

| ID | Семейство / carrier | Механика | Тип источника | Профили | Захваты | Размеченные строки |
|---|---|---|---|---|---:|---:|
| CC_BODY_01 | body / json | direct synthetic command/result JSON | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BODY_02 | body / json_junk | payload among junk JSON fields | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BODY_03 | body / form | fragments in form values | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BODY_04 | body / multipart | synthetic file-like result | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BODY_05 | body / multipart_filename | data in multipart filename/boundary | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BODY_06 | body / octet_stream | encrypted-like bytes | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BODY_07 | body / protobuf_like | opaque protobuf-like bytes | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BODY_08 | body / xml_attributes | payload in XML attributes | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BODY_09 | body / graphql_variables | command disguised as variables | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BODY_10 | body / chunked | fragmented HTTP body | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BROWSER_01 | browser / extension_registration | extension registration | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BROWSER_02 | browser / service_worker_alarm | Service Worker alarm | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BROWSER_03 | browser / push_wakeup | push-like wake-up | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BROWSER_04 | browser / keepalive_tab | hidden helper fetch | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BROWSER_05 | browser / native_echo | safe local echo bridge | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BROWSER_06 | browser / browser_wss | WSS from browser-like context | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_BROWSER_07 | browser / headless | headless browser timing | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BROWSER_08 | browser / fetch_stream | synthetic blob fetch stream | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BROWSER_09 | browser / send_beacon | sendBeacon-like exfil | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BROWSER_10 | browser / csp_report | CSP report carrier | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_BROWSER_11 | browser / prefetch | resource hint subdomain symbols | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_CONNECT_01 | connect / http_connect | HTTP CONNECT-like local echo fixture | ограниченный пример семантики | original_dispatch | 2 | 24 |
| CC_CONNECT_02 | connect / extended_connect | H2 Extended CONNECT semantic fixture | ограниченный пример семантики | original_dispatch | 2 | 4 |
| CC_CONNECT_03 | connect / websocket_h2 | WebSocket-over-H2 semantic fixture | ограниченный пример семантики | original_dispatch | 2 | 4 |
| CC_DOH_01 | doh / dns_message | DNS wire-format in local /dns-query | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_GRPC_01 | grpc / grpc_unary | gRPC wire-format unary fixture | ограниченный пример семантики | original_dispatch | 2 | 4 |
| CC_GRPC_02 | grpc / grpc_server_stream | server stream fixture | ограниченный пример семантики | original_dispatch | 2 | 4 |
| CC_GRPC_03 | grpc / grpc_client_stream | client stream fixture | ограниченный пример семантики | original_dispatch | 2 | 4 |
| CC_GRPC_04 | grpc / grpc_bidi | bidi stream fixture | ограниченный пример семантики | original_dispatch | 2 | 4 |
| CC_GRPC_05 | grpc / grpc_metadata | metadata carrier | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_GRPC_06 | grpc / grpc_trailers | status/trailer signal | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_GRPC_07 | grpc / grpc_compressed | compressed messages | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_GRPC_08 | grpc / grpc_multiplex | multiplexed methods | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_H2_01 | http2 / h2_request | GET/POST streams | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_H2_02 | http2 / h2_parallel | parallel stream fragments | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_H2_03 | http2 / h2_order | stream ordering channel | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_H2_04 | http2 / h2_header | header value carrier | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_H2_05 | http2 / h2_data_size | DATA size classes | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_H2_06 | http2 / h2_reset | selective resets fixture | ограниченный пример семантики | original_dispatch | 2 | 24 |
| CC_H2_07 | http2 / h2_sparse | sparse streams on long connection | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_H2_08 | http2 / h2_client_mismatch | browser-like headers from script stack | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_H3_01 | http3 / h3_request | HTTP/3 GET/POST streams | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_H3_02 | http3 / h3_parallel | parallel H3 stream fragments | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_H3_03 | http3 / h3_order | stream ordering channel | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_H3_04 | http3 / h3_header | H3 header value carrier | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_H3_05 | http3 / h3_data_size | H3 DATA size classes | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_H3_06 | http3 / h3_sparse | sparse H3 streams | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_H3_07 | http3 / h3_reconnect | QUIC reconnect/NAT-rebinding-like profile | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_H3_08 | http3 / h3_fallback | H3 to H2 to H1 fallback campaign | генератор протокольного обмена | original_dispatch | 2 | 12 |
| CC_HDR_01 | header / cookie | registration in Cookie | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_HDR_02 | header / cookie | fragments in Cookie | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_HDR_03 | header / authorization | synthetic payload disguised as bearer | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_HDR_04 | header / user_agent | ID suffix in User-Agent | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_HDR_05 | header / referer | encoded result in Referer | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_HDR_06 | header / origin | low bandwidth marker in Origin | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_HDR_07 | header / accept_language | Accept-Language order modulation | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_HDR_08 | header / accept_encoding | Accept-Encoding order modulation | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_HDR_09 | header / if_none_match | synthetic victim ID in If-None-Match | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_HDR_10 | header / if_modified_since | timestamp modulation | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_HDR_11 | header / range | Range offsets encode symbols | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_HDR_12 | header / content_type_param | boundary/charset fragment | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_LOTS_01 | lots / chatops_poll | local getUpdates-like polling | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_LOTS_02 | lots / chatops_push | local sendMessage-like request | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_LOTS_03 | lots / chatops_document | local sendDocument-like multipart | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_LOTS_04 | lots / webhook | local webhook-like one-way POST | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_LOTS_05 | lots / ephemeral_relay | local ephemeral relay analogue | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_LOTS_06 | lots / paas_endpoint | local PaaS-style endpoint | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_LOTS_07 | lots / dyndns_endpoint | local dynamic-DNS-style endpoint | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_LOTS_08 | lots / bucket_dead_drop | local S3-like time-slot object name | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_LP_01 | longpoll / long_poll_recv | long polling receive | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_LP_02 | longpoll / rotating_poll | rotating poll ID | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_LP_03 | longpoll / poll_upload | poll + result upload | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_MASQUE_01 | masque / connect_udp | real H3 datagram CONNECT-UDP-like fixture | ограниченный пример семантики | original_dispatch | 2 | 4 |
| CC_MASQUE_02 | masque / connect_ip | CONNECT-IP-like isolated synthetic datagram fixture | ограниченный пример семантики | original_dispatch | 2 | 4 |
| CC_MQTT_01 | mqtt_ws / mqtt_topic | MQTT-like CONNECT/PUBLISH frames over WSS | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_MQTT_02 | mqtt_ws / mqtt_payload | MQTT-like topic/payload carrier | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_MQTT_03 | mqtt_ws / mqtt_timing | MQTT-over-WSS timing fixture | ограниченный пример семантики | original_dispatch | 2 | 4 |
| CC_OHTTP_01 | privacy / ohttp_binary | OHTTP-like encrypted binary benign hard-negative | ограниченный пример семантики | original_dispatch | 2 | 4 |
| CC_RESP_01 | response / response_body | plain task in response | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_RESP_02 | response / response_json | task in noisy object | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_RESP_03 | response / response_xml | GUID/hex steganography | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_RESP_04 | response / response_html | HTML comment/attribute carrier | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_RESP_05 | response / response_css | CSS value carrier | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_RESP_06 | response / response_image_meta | synthetic image metadata carrier | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_RESP_07 | response / set_cookie | downstream command in Set-Cookie | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_RESP_08 | response / etag | downstream ETag fragment | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_RESP_09 | response / status_code | status-code alphabet | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_RESP_10 | response / redirect | Location carries task | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_RESP_11 | response / response_size | response length modulation | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_RESP_12 | response / response_chunks | chunk-size modulation | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_SSE_01 | sse / sse_command | command event stream | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_SSE_02 | sse / sse_encoded | encoded events | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_SSE_03 | sse / sse_comment_timing | comment timing | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_SSE_04 | sse / last_event_id | Last-Event-ID carrier | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_SSE_05 | sse / sse_sparse | sparse long-lived SSE | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_SYN_01 | syntax / header_order | header order channel | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_SYN_02 | syntax / header_case | header case channel | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_SYN_03 | syntax / whitespace | OWS modulation | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_SYN_04 | syntax / duplicate_headers | duplicate header order | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_SYN_05 | syntax / method | method selection channel | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_SYN_06 | syntax / connection_reuse | reuse/close modulation | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_SYN_07 | syntax / content_length_parity | body length parity | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_SYN_08 | syntax / request_count | request count symbols | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_TIME_01 | timing / fixed_beacon | fixed beacon | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_TIME_02 | timing / low_jitter | low jitter beacon | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_TIME_03 | timing / medium_jitter | medium jitter beacon | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_TIME_04 | timing / high_jitter | lognormal jitter | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_TIME_05 | timing / burst | burst then pause | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_TIME_06 | timing / work_hours | work-hours schedule | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_TIME_07 | timing / backoff | exponential backoff | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_TIME_08 | timing / binary_timing | short/long interval bits | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_TIME_09 | timing / low_slow | accelerated low-and-slow fixture | ограниченный пример семантики | original_dispatch | 2 | 24 |
| CC_TIME_10 | timing / event_driven | event-driven callback | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_TLS_01 | tls / script_default | default script TLS | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_TLS_02 | tls / browser_like | browser-like headers/TLS challenge | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_TLS_03 | tls / clienthello_variation | TLS stack diversity challenge | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_TLS_04 | tls / resumption | session resumption | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_TLS_05 | tls / zero_rtt_marker | 0-RTT capability marker fixture | ограниченный пример семантики | original_dispatch | 2 | 24 |
| CC_TLS_06 | tls / sni_loss_fixture | visibility-loss fixture | ограниченный пример семантики | original_dispatch | 2 | 24 |
| CC_TLS_07 | tls / fallback | H3/H2/H1 metadata fallback fixture | ограниченный пример семантики | original_dispatch | 2 | 24 |
| CC_TLS_08 | tls / shared_edge | shared-edge hostname categories | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_TLS_09 | tls / cert_rotation | certificate/destination rotation fixture | ограниченный пример семантики | original_dispatch | 2 | 24 |
| CC_TLS_10 | tls / inspection_bypass | pinning-like inspection outcome label | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_TUN_01 | tunnel / fixed_echo_relay | fixed local echo service | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_TUN_02 | tunnel / allowlisted_connect | SOCKS-like grammar to synthetic target only | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_TUN_03 | tunnel / multiplex_conn_id | 2-16 logical synthetic streams | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_TUN_04 | tunnel / mixed_inner | synthetic inner protocol labels | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_TUN_05 | tunnel / backpressure | safe window/update-like controls | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_TUN_06 | tunnel / stream_lifecycle | half-close/reset grammar | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_TUN_07 | tunnel / interactive | small bidirectional messages | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_TUN_08 | tunnel / bulk | larger one-way synthetic phase | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_TUN_09 | tunnel / keepalive | idle + sparse ping/pong | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_TUN_10 | tunnel / endpoint_migration | same campaign second local hostname | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_URI_01 | uri / query_value | message in id query | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_URI_02 | uri / query_values | message split over query values | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_URI_03 | uri / path_segment | fragment in path segment | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_URI_04 | uri / multi_request_query | small fragments over requests | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_URI_05 | uri / filename | payload-like filename extension | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_URI_06 | uri / matrix_path | matrix-like path params | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_URI_07 | uri / query_key | symbols in query key names | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_URI_08 | uri / uri_length | URI length modulation | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_WEBTRANS_01 | webtransport / webtransport_stream | real WebTransport bidirectional stream fixture | ограниченный пример семантики | original_dispatch | 2 | 4 |
| CC_WS_01 | websocket / ws_registration | UUID + synthetic system profile | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_WS_02 | websocket / ws_json_recv | JSON recv polling | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_WS_03 | websocket / ws_json_send | JSON result send | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_WS_04 | websocket / ws_b64_envelope | Base64 JSON envelope | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_WS_05 | websocket / ws_binary | binary frames | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_WS_06 | websocket / ws_ping | ping timing/payload | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_WS_07 | websocket / ws_fragmentation | fragmented messages | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_WS_08 | websocket / ws_idle | long-lived sparse messages | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_WS_09 | websocket / ws_reconnect | reconnect continuity | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_WS_10 | websocket / ws_multipersona | multiple personas same endpoint | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_WS_11 | websocket / ws_browser_headers | browser mimicry | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_WS_12 | websocket / ws_appliance | appliance-like WSS | генератор протокольного обмена | original_dispatch | 2 | 4 |
| CC_XHDR_01 | custom_header / x_session_id | X-Session-Id value stream | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_XHDR_02 | custom_header / x_request_id | X-Request-Id value stream | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_XHDR_03 | custom_header / x_correlation_id | correlation ID stream | генератор протокольного обмена | original_dispatch | 2 | 24 |
| CC_XHDR_04 | custom_header / x_telemetry | encoded telemetry-like result | генератор протокольного обмена | original_dispatch | 2 | 24 |
| M-CLOUD-API | M-CLOUD-API / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | cloud-api-python_httpx-0, cloud-api-python_httpx_h2-1, cloud-api-python_stdlib-2, cloud-api-curl_linux-3, cloud-api-go_nethttp-4, cloud-api-node_fetch-5, cloud-api-java_httpclient-6, cloud-api-rust_reqwest-7, cloud-api-browser_chromium-8 | 18 | 480 |
| M-DEAD-DROP | M-DEAD-DROP / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | m-dead-drop-python_httpx-0, m-dead-drop-python_httpx_h2-1, m-dead-drop-python_stdlib-2, m-dead-drop-curl_linux-0, m-dead-drop-go_nethttp-1, m-dead-drop-node_fetch-2 | 12 | 48 |
| M-DNS-BEACON | M-DNS-BEACON / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | dns-python-auth-udp, dns-python-auth-tcp, dns-python-rec-udp, dns-python-rec-tcp | 8 | 96 |
| M-DNS-BULK | M-DNS-BULK / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | dns-python-auth-udp, dns-python-auth-tcp, dns-python-rec-udp, dns-python-rec-tcp | 8 | 96 |
| M-DOH | M-DOH / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | doh-python_httpx-direct, doh-python_httpx_h2-nginx, doh-python_stdlib-direct, doh-curl_linux-nginx, doh-go_nethttp-direct, doh-node_fetch-nginx | 12 | 144 |
| M-DOQ | M-DOQ / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | doq-aioquic-newconn, doq-aioquic-reuse | 4 | 28 |
| M-FALLBACK | M-FALLBACK / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | m-fallback-python_httpx-0, m-fallback-python_httpx_h2-1, m-fallback-python_stdlib-2, m-fallback-curl_linux-0, m-fallback-go_nethttp-1, m-fallback-node_fetch-2 | 12 | 144 |
| M-GRPC-BIDI | M-GRPC-BIDI / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | grpc-python-bidi, grpc-python-stream-unary | 4 | 8 |
| M-H3-QUIC | M-H3-QUIC / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | h3-aioquic-newconn, h3-aioquic-reuse, h3-aioquic-parallel | 6 | 32 |
| M-HTTP-443 | M-HTTP-443 / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | http443-python_httpx-nginx, http443-python_httpx_h2-nginx, http443-python_stdlib-nginx, http443-curl_linux-nginx, http443-go_nethttp-nginx, http443-node_fetch-nginx | 12 | 144 |
| M-HTTPS-BEACON | M-HTTPS-BEACON / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | m-https-beacon-python_httpx-hypercorn, m-https-beacon-python_httpx_h2-nginx_reverse_proxy, m-https-beacon-python_stdlib-hypercorn, m-https-beacon-curl_linux-nginx_reverse_proxy, m-https-beacon-go_nethttp-hypercorn, m-https-beacon-node_fetch-nginx_reverse_proxy, m-https-beacon-java_httpclient-hypercorn, m-https-beacon-rust_reqwest-nginx_reverse_proxy, m-https-beacon-browser_chromium-hypercorn, m-https-beacon-python_httpx_reuse-nginx_reverse_proxy, m-https-beacon-python_httpx_h2_reuse-hypercorn | 22 | 280 |
| M-HTTPS-FRAG | M-HTTPS-FRAG / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | m-https-frag-python_httpx-hypercorn, m-https-frag-python_httpx_h2-nginx_reverse_proxy, m-https-frag-python_stdlib-hypercorn, m-https-frag-curl_linux-nginx_reverse_proxy, m-https-frag-go_nethttp-hypercorn, m-https-frag-node_fetch-nginx_reverse_proxy, m-https-frag-java_httpclient-hypercorn, m-https-frag-rust_reqwest-nginx_reverse_proxy, m-https-frag-browser_chromium-hypercorn, m-https-frag-python_httpx_reuse-nginx_reverse_proxy, m-https-frag-python_httpx_h2_reuse-hypercorn | 22 | 248 |
| M-HTTPS-FRONT | M-HTTPS-FRONT / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | m-https-front-python_httpx-0, m-https-front-python_httpx_h2-1, m-https-front-python_stdlib-2, m-https-front-curl_linux-3, m-https-front-go_nethttp-4, m-https-front-node_fetch-5, m-https-front-java_httpclient-6, m-https-front-rust_reqwest-7, m-https-front-browser_chromium-8, m-https-front-python_httpx_reuse-9, m-https-front-python_httpx_h2_reuse-10 | 22 | 248 |
| M-HTTPS-LOWENT | M-HTTPS-LOWENT / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | m-https-lowent-python_httpx-hypercorn, m-https-lowent-python_httpx_h2-nginx_reverse_proxy, m-https-lowent-python_stdlib-hypercorn, m-https-lowent-curl_linux-nginx_reverse_proxy, m-https-lowent-go_nethttp-hypercorn, m-https-lowent-node_fetch-nginx_reverse_proxy, m-https-lowent-java_httpclient-hypercorn, m-https-lowent-rust_reqwest-nginx_reverse_proxy, m-https-lowent-browser_chromium-hypercorn, m-https-lowent-python_httpx_reuse-nginx_reverse_proxy, m-https-lowent-python_httpx_h2_reuse-hypercorn | 22 | 248 |
| M-L34-STORAGE | M-L34-STORAGE / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | raw-ipv4-id-udp, raw-udp-source-port, raw-icmp-id-seq, raw-tcp-initial-seq, raw-tcp-timestamp | 10 | 76 |
| M-PUBSUB-MQTT | M-PUBSUB-MQTT / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | mqtt-paho-qos0, mqtt-paho-qos1 | 4 | 8 |
| M-RMM-SHAPE | M-RMM-SHAPE / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | m-rmm-shape-python_httpx-hypercorn, m-rmm-shape-python_httpx_h2-nginx_reverse_proxy, m-rmm-shape-python_stdlib-hypercorn, m-rmm-shape-curl_linux-nginx_reverse_proxy, m-rmm-shape-go_nethttp-hypercorn, m-rmm-shape-node_fetch-nginx_reverse_proxy, m-rmm-shape-java_httpclient-hypercorn, m-rmm-shape-rust_reqwest-nginx_reverse_proxy, m-rmm-shape-browser_chromium-hypercorn, m-rmm-shape-python_httpx_reuse-nginx_reverse_proxy, m-rmm-shape-python_httpx_h2_reuse-hypercorn | 22 | 280 |
| M-TIMING-XCARRIER | M-TIMING-XCARRIER / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | timing-https-curl, timing-https-go, timing-https-browser, timing-dns-udp, timing-dns-tcp, timing-wss-python, timing-wss-node, timing-mqtt-paho | 16 | 156 |
| M-TUNNEL | M-TUNNEL / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | wss-python-direct, wss-python-nginx, wss-node-nginx, wss-chromium-nginx | 8 | 24 |
| M-WSS-LONG | M-WSS-LONG / implementation_specific | Описание в реестре отсутствует; см. фактические реализации ниже | генератор формы активности | wss-python-direct, wss-python-nginx, wss-node-nginx, wss-chromium-nginx | 8 | 24 |
| ADAPTIX_GOPHER_TCP | framework_task_result / tcp | Upstream Linux Gopher registration + six synthetic cat tasks/results; not HTTP or DNS | настоящий upstream framework | p0, p1, p2 | 6 | 12 |
| ADAPTIX_GOPHER_MTLS | framework_task_result / mtls | Upstream Linux Gopher registration + six synthetic cat tasks/results; not HTTP or DNS | настоящий upstream framework | p0, p1, p2 | 6 | 12 |

### Фактически наблюдавшиеся реализации Stage M

| ID | Transport / carrier / поле / вид источника |
|---|---|
| M-CLOUD-API | https / cloud_api / application_message — генератор формы активности |
| M-DEAD-DROP | https+https / dead_drop / session_sequence — генератор формы активности |
| M-DNS-BEACON | dns / dns_beacon / dns_message — генератор формы активности |
| M-DNS-BULK | dns / dns_bulk / dns_message — генератор формы активности |
| M-DOH | https+doh / doh / dns_message — генератор формы активности |
| M-DOQ | doq+quic / doq / dns_message — генератор формы активности |
| M-FALLBACK | https+dns / fallback / session_sequence — генератор формы активности |
| M-GRPC-BIDI | grpc+h2 / grpc_bidi / application_message — генератор формы активности |
| M-H3-QUIC | h3+quic / h3_quic / application_message — генератор формы активности |
| M-HTTP-443 | http / http_443 / application_message — генератор формы активности |
| M-HTTPS-BEACON | https / https_beacon / session_sequence — генератор формы активности |
| M-HTTPS-FRAG | https / https_frag / application_message — генератор формы активности |
| M-HTTPS-FRONT | https / https_front / application_message — генератор формы активности |
| M-HTTPS-LOWENT | https / https_lowent / application_message — генератор формы активности |
| M-L34-STORAGE | icmp / l34_storage / network_header — генератор формы активности; ipv4+udp / l34_storage / network_header — генератор формы активности; tcp_raw / l34_storage / network_header — генератор формы активности |
| M-PUBSUB-MQTT | mqtt+wss / pubsub_mqtt / application_message — генератор формы активности |
| M-RMM-SHAPE | https / rmshape / session_sequence — генератор формы активности |
| M-TIMING-XCARRIER | dns / timing_xcarrier / session_sequence — генератор формы активности; https / timing_xcarrier / session_sequence — генератор формы активности; mqtt+wss / timing_xcarrier / session_sequence — генератор формы активности; wss / timing_xcarrier / session_sequence — генератор формы активности |
| M-TUNNEL | wss / tunnel / application_message — генератор формы активности |
| M-WSS-LONG | wss / wss_long / session_sequence — генератор формы активности |

## 7. Все 89 признаков

Их значения не подгонялись под офис. В ноутбуке указан словарь; точные вычисления остаются в общем коде извлечения.

| Признак | Группа | Значение |
|---|---|---|
| pkt_count | объём, темп и форма потока | пакетов в сессии |
| up_pkt_count | объём, темп и форма потока | пакетов от клиента |
| down_pkt_count | объём, темп и форма потока | пакетов от сервера |
| total_bytes | объём, темп и форма потока | байт всего (на проводе) |
| up_bytes | объём, темп и форма потока | байт от клиента |
| down_bytes | объём, темп и форма потока | байт от сервера |
| flow_duration | объём, темп и форма потока | длительность сессии, с |
| up_down_pkt_ratio | объём, темп и форма потока | пакеты клиент/сервер |
| up_down_bytes_ratio | объём, темп и форма потока | байты клиент/сервер |
| pkt_rate | объём, темп и форма потока | пакетов в секунду |
| byte_rate | объём, темп и форма потока | байт в секунду |
| pkt_len_mean | размеры пакетов | длина пакета: среднее |
| pkt_len_std | размеры пакетов | длина пакета: разброс |
| pkt_len_min | размеры пакетов | длина пакета: минимум |
| pkt_len_max | размеры пакетов | длина пакета: максимум |
| pkt_len_median | размеры пакетов | длина пакета: медиана (ближайший элемент ряда, без интерполяции) |
| pkt_len_p10 | размеры пакетов | длина пакета: перцентиль 10 (ближайший элемент ряда, без интерполяции) |
| pkt_len_p90 | размеры пакетов | длина пакета: перцентиль 90 (ближайший элемент ряда, без интерполяции) |
| pkt_len_p95 | размеры пакетов | длина пакета: перцентиль 95 (ближайший элемент ряда, без интерполяции) |
| pkt_len_entropy | размеры пакетов | длина пакета: энтропия |
| iat_mean | интервалы между пакетами | интервал между пакетами: среднее |
| iat_std | интервалы между пакетами | интервал между пакетами: разброс |
| iat_min | интервалы между пакетами | интервал между пакетами: минимум |
| iat_max | интервалы между пакетами | интервал между пакетами: максимум |
| iat_p50 | интервалы между пакетами | интервал между пакетами: перцентиль 50 (ближайший элемент ряда, без интерполяции) |
| iat_p90 | интервалы между пакетами | интервал между пакетами: перцентиль 90 (ближайший элемент ряда, без интерполяции) |
| iat_p99 | интервалы между пакетами | интервал между пакетами: перцентиль 99 (ближайший элемент ряда, без интерполяции) |
| iat_entropy | интервалы между пакетами | интервал между пакетами: энтропия |
| iat_cv | интервалы между пакетами | интервал между пакетами: коэфф. вариации |
| iat_regularity | интервалы между пакетами | регулярность интервалов |
| burst_count | пачки пакетов | пачек по времени: сколько раз подряд шли пакеты с интервалом меньше 50 мс (в любую сторону) |
| idle_ratio | объём, темп и форма потока | доля времени простоя |
| direction_changes | объём, темп и форма потока | смены направления |
| syn_count | TCP-флаги | число пакетов с флагом SYN |
| fin_count | TCP-флаги | число пакетов с флагом FIN |
| rst_count | TCP-флаги | число пакетов с флагом RST |
| psh_count | TCP-флаги | число пакетов с флагом PSH |
| ack_count | TCP-флаги | число пакетов с флагом ACK |
| len_entropy_up | размеры пакетов | энтропия длин от клиента |
| len_entropy_down | размеры пакетов | энтропия длин от сервера |
| len_unique_up | объём, темп и форма потока | различных длин от клиента |
| len_unique_down | объём, темп и форма потока | различных длин от сервера |
| small_pkt_share | объём, темп и форма потока | доля кадров короче 100 байт (включая пустые подтверждения TCP) |
| low_rate_long | объём, темп и форма потока | длинная и медленная сессия |
| dir_entropy | объём, темп и форма потока | энтропия направлений |
| idle_gt60_count | объём, темп и форма потока | простоев дольше 60 с |
| up_bytes_per_pkt | объём, темп и форма потока | байт на пакет от клиента |
| down_bytes_per_pkt | объём, темп и форма потока | байт на пакет от сервера |
| const_len_share_up | объём, темп и форма потока | доля одинаковых длин от клиента |
| burst_len_mean | пачки пакетов | средняя длина пачки по времени, пакетов (0 — пачек не было) |
| burst_len_max | пачки пакетов | самая длинная пачка по времени, пакетов |
| dir_burst_count | пачки пакетов | пачек по направлению: сколько раз подряд шли 2 и больше пакетов в одну сторону (время не учитывается) |
| dir_burst_len_mean | пачки пакетов | средняя длина пачки по направлению, пакетов |
| dir_burst_len_max | пачки пакетов | самая длинная пачка по направлению, пакетов |
| ra_small_up_count | удалённое администрирование: похоже ли на человека, работающего в удалённой консоли | «нажатий клавиш»: пакетов клиента с короткой порцией данных (1–128 байт); пустые подтверждения TCP не считаются |
| ra_small_up_share | удалённое администрирование: похоже ли на человека, работающего в удалённой консоли | доля «нажатий» среди всех пакетов клиента с данными (1 — клиент шлёт только короткие команды) |
| ra_keystroke_iat_p50 | удалённое администрирование: похоже ли на человека, работающего в удалённой консоли | медианная пауза между соседними «нажатиями», с (у человека за терминалом — десятые доли секунды) |
| ra_echo_ratio | удалённое администрирование: похоже ли на человека, работающего в удалённой консоли | доля «нажатий», на которые сервер ответил данными в пределах 1 с (эхо набранного символа); пусто, если ответы сервера зеркалу не видны |
| ra_echo_latency_p50 | удалённое администрирование: похоже ли на человека, работающего в удалённой консоли | медианная задержка ответа сервера на «нажатие», с; пусто, если ответы сервера не видны |
| ra_interactive_score | удалённое администрирование: похоже ли на человека, работающего в удалённой консоли | признак интерактивной работы: доля «нажатий», если их не меньше 10 и медианное эхо быстрее 0,2 с, иначе 0; пусто, если ответы сервера не видны |
| ra_burst_after_idle | удалённое администрирование: похоже ли на человека, работающего в удалённой консоли | сколько раз активность возобновлялась после паузы дольше минуты |
| tcp_handshake_rtt_ms | контекст сессии: стороны, рукопожатие, сервис, время | время от SYN клиента до SYN/ACK сервера, мс (сетевая задержка до сервера); пусто, если не видно |
| data_pkt_up | контекст сессии: стороны, рукопожатие, сервис, время | число пакетов клиента с данными (длина полезной нагрузки TCP/UDP > 0); чистые подтверждения не считаются |
| data_pkt_down | контекст сессии: стороны, рукопожатие, сервис, время | число пакетов сервера с данными (длина полезной нагрузки > 0) |
| small_data_up_bytes | контекст сессии: стороны, рукопожатие, сервис, время | сумма байт данных в коротких пакетах клиента (данные 1–128 байт) — объём «набранного руками» |
| pay_entropy_up | содержимое (первые байты данных) | энтропия первых байт данных от клиента |
| pay_entropy_down | содержимое (первые байты данных) | энтропия первых байт данных от сервера |
| pay_printable_up | содержимое (первые байты данных) | доля печатаемых символов от клиента |
| pay_printable_down | содержимое (первые байты данных) | доля печатаемых символов от сервера |
| pay_null_share_up | содержимое (первые байты данных) | доля нулевых байт от клиента |
| pay_b64_share_up | содержимое (первые байты данных) | доля символов base64 от клиента |
| pay_bytes_sampled_up | содержимое (первые байты данных) | сколько байт данных клиента разобрано |
| pay_bytes_sampled_down | содержимое (первые байты данных) | сколько байт данных сервера разобрано |
| tcp_retx_pkts_up | повторные передачи TCP | повторно отправленные клиентом пакеты с данными (номер TCP уже был передан; проверки keep-alive исключены); пусто для UDP |
| tcp_retx_pkts_down | повторные передачи TCP | повторно отправленные сервером пакеты с данными; пусто для UDP |
| tcp_retx_bytes_up | повторные передачи TCP | байт данных в повторах клиента |
| tcp_retx_bytes_down | повторные передачи TCP | байт данных в повторах сервера |
| tls_sni_len | TLS | длина SNI |
| tls_version | TLS | версия TLS |
| tls_cipher_count | TLS | шифров предложено |
| tls_ext_count | TLS | расширений TLS |
| tls_group_count | TLS | групп ключевого обмена |
| tls_sigalg_count | TLS | алгоритмов подписи |
| tls_alpn_h2 | TLS | ALPN = HTTP/2 |
| tls_alpn_h3 | TLS | ALPN = HTTP/3 |
| tls_alpn_http11 | TLS | ALPN = HTTP/1.1 |
| tls_alpn_other | TLS | другой ALPN |
| tls_resumed | TLS | возобновление TLS-сессии |
| tls_early_data | TLS | 0-RTT |

## 8. Что обучалось и что означает качество

Исследовательский классификатор различает сценарии и их контроли на закреплённых группах профилей. Офис без истинной метки не участвует как отрицательный класс. Его test ROC-AUC измеряет распознавание отложенных профилей этого корпуса, а не FPR или переносимость в реальном офисе.

Кластеры обучаются отдельно без целевой метки. Диагностика происхождения решает третью задачу — office против generated-control. Эти три результата отвечают на разные вопросы.

Сценарий/контроль, все 89 входных признаков, test: `{'roc_auc': 0.7225811334759212, 'average_precision': 0.6728647172884283, 'balanced_accuracy_at_050': 0.6579038606154717, 'rows': 1292}`. Split-строки: `{'train': 3426, 'test': 1292, 'validation': 1306}`. Малая доля независимых Adaptix-профилей ограничивает выводы даже при хорошем сегментном ROC-AUC.


## 8.1. Parquet и воспроизводимое обучение

Локальный каталог: `../data/cover_channels_natural_public_20261006`. Таблица `cover_channels_research_features.parquet` воспроизводит популяцию графика и диагностики. `cover_channels_full_background_features.parquet` содержит тот же размеченный трафик и полный сохранённый фон; у полного офиса split/cluster пустые. Офисные label_binary остаются null.

Для X используйте **только** `feature_columns.json`. Для точного повторения группового протокола берите оригиналы `source/features_all.parquet` и `source/labels_metadata.parquet`. Разбиение профилей закреплено до fit; обе роли и все повторные размещения профиля идут в один split. Старое native-разбиение сохранено, новые Adaptix-группы добавлены отдельно.

Нельзя превращать весь офис в отрицательный класс или делать случайный split строк: это создаёт неверную разметку и утечку между частями одного запуска.


In [ ]:
from pathlib import Path
import os
import json
import pandas as pd

data = Path(os.environ['TRAFFIC_RESEARCH_DATA'])
# Original private research export: feature_columns.json + source/*.parquet.
names = json.loads((data / "feature_columns.json").read_text())
X = pd.read_parquet(data / "source/features_all.parquet")[names]
metadata = pd.read_parquet(data / "source/labels_metadata.parquet")
assert len(names) == 89 and len(X) == len(metadata)
train = metadata["split"].eq("train") & metadata["label_binary"].notna()
test = metadata["split"].eq("test") & metadata["label_binary"].notna()
X_train, y_train = X.loc[train], metadata.loc[train, "label_binary"].astype(int)
X_test, y_test = X.loc[test], metadata.loc[test, "label_binary"].astype(int)
print(X.shape, metadata.groupby(["split", "label_state"]).size())
# Сюда можно подключить обучение; офисные строки не объявлены отрицательными.


## 9. Что ещё ограничивает результат

- Two retained office days, discontinuous blocks; repeated campaigns do not add independent executions.
- MTU/RTT and timestamp-observation calibration is conditional on these measured backgrounds, not a new unseen-office validation.
- Office rows remain unlabelled; origin AUC is not FPR.
- Complete equivalence of TLS fingerprints, TCP option order, window scale and server TTL has not been established; these are not all columns of the 89-feature X.
- Activity shapes are not edited to make a classifier fail.
- Public captures include an additional emulated path delay on top of the actual proxy path; equivalence of full office latency is unverified.
- Additional microsecond readout is an empirically bounded approximation; not a reproduced hardware capture mechanism.
- Public page captures use an existing unauthenticated HTTP proxy; logical endpoint mapping does not prove a direct private-to-public path.
- Added ordinary HTTPS fixtures broaden observed client behavior; do not establish equivalent application mix or independent office-positive transfer.

Следующая независимая проверка — позитивы, действительно увиденные офисным сенсором. Стендовое выравнивание пути само по себе не исправляет измеренный FPR в живом офисе.


## 10. Проверяемые источники

- [cover_channels_natural_public_20261006.json](../reports/cover_channels_natural_public_20261006.json); SHA256 `53fefa187808cad80bd52d06ec8a5ade1870acd8e52a590c5398370604636be1`
- [cover_channels_natural_clusters_public_20261006.json](../reports/cover_channels_natural_clusters_public_20261006.json); SHA256 `44102bfb727e5411a6d8d443afc3eba553f91f621f36acc3c9568b93227465e2`
- [cover_channels_natural_projection_public_20261006.json](../reports/cover_channels_natural_projection_public_20261006.json); SHA256 `46eba57a74722861eeac3c76e229253183c3ecdd441fcde5fa0f94976377fecd`
- [cover_channels_origin_power_native_public_20261006.json](../reports/cover_channels_origin_power_native_public_20261006.json); SHA256 `cc4ab16d5b486150deef7d74bec6b4bfcf5991987514a2271360a04dcd0451c2`
- [cover_channels_feature_tradeoff_public_20261006.json](../reports/cover_channels_feature_tradeoff_public_20261006.json); SHA256 `8eff4cd5f1be73585db4cc5dc74c52f892d1d54fde8faef3df655702d5665326`
- [cover_channels_comparable_origin_public_20261006.json](../reports/cover_channels_comparable_origin_public_20261006.json); SHA256 `6daedb7c0a7d95e8e7d2f3f19c473113de94606473045854cefe4ac264ac30f0`

Training COMPLETE: `4c16b604c854de5086d54eed3dec217fb6d39a764c7f15f7f1d73cf4459ca524`. Source identity: `c852fdee8f9668be9097b7ea0cef7ad9363a9456d7fc8fbe354e70ebae5ec176`.
